# StudyAbroadGPT LoRA Downstream Evaluation

This Google Colab notebook evaluates the fine-tuned merged model:

`millat/StudyAbroadGPT-7B-LoRa-Kaggle` with `subfolder="merged"`

against its base model loaded from:

`mistralai/Mistral-7B-Instruct-v0.3`

using held-out prompts from the `test` split of:

`millat/StudyAbroadGPT-Dataset`

The notebook is designed for **Google Colab with a T4 GPU** and focuses only on evaluation/inference. It does **not** contain training code.

## Evaluation design

- Load 50 random test conversations with fixed seed = 42.
- Extract the first user prompt from each conversation.
- Generate one response from the base model and one response from the LoRA/merged model.
- Use deterministic generation for the scored benchmark.
- Randomize A/B ordering for blinded human scoring.
- Export a blinded CSV to `/content/outputs/downstream_blinded_evaluation.csv`.
- Save generation metadata and reproducibility config.
- Provide post-scoring analysis utilities.

## Important note

The scored evaluation uses deterministic generation:

- `do_sample=False`
- `temperature=0.0`

An optional sampling/demo config is included for qualitative exploration, but the exported evaluation CSV uses deterministic mode for reproducibility.

## 1. Install dependencies

In [ ]:
!pip install -q transformers accelerate datasets bitsandbytes torch pandas tqdm sentencepiece peft

In [ ]:
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_output_dir = '/content/drive/MyDrive/LoRA_Paper/outputs'
    os.environ['DRIVE_OUTPUT_DIR'] = drive_output_dir
    Path(drive_output_dir).mkdir(parents=True, exist_ok=True)
    print('Drive outputs:', drive_output_dir)
except Exception as exc:
    print('Google Drive mount skipped:', exc)

## 1b. (Optional) Mount Google Drive for outputs

If running in Colab, mount Drive and save all outputs under your Drive.

## 2. GPU verification

This section verifies CUDA availability, GPU name, and approximate VRAM. A Colab T4 runtime is recommended.

In [ ]:
import os
import gc
import json
import time
import random
from datetime import datetime, timezone
from pathlib import Path

import torch
import pandas as pd
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

VERBOSE = False
SHOW_PREVIEW = False

def log(msg: str):
    if VERBOSE:
        print(msg)

def resolve_output_dir():
    drive_dir = os.environ.get('DRIVE_OUTPUT_DIR')
    if drive_dir:
        return Path(drive_dir)
    if os.environ.get('COLAB_RELEASE_TAG'):
        return Path('/content/outputs')
    return Path('outputs')

OUTPUT_DIR = resolve_output_dir()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def load_csv_if_exists(path: Path):
    if path.exists():
        return pd.read_csv(path)
    return None

def save_df(df: pd.DataFrame, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    log(f'Saved: {path}')

def load_json_if_exists(path: Path):
    if path.exists():
        with open(path, 'r') as f:
            return json.load(f)
    return None

def save_json(data: dict, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w') as f:
        json.dump(data, f, indent=2)
    log(f'Saved: {path}')

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    total_vram_gb = props.total_memory / (1024**3)
    free_mem, total_mem = torch.cuda.mem_get_info()
    log(f'GPU: {gpu_name}')
    log(f'Total VRAM: {total_vram_gb:.2f} GB')
    log(f'Currently free VRAM: {free_mem / (1024**3):.2f} GB')
else:
    raise RuntimeError('CUDA is not available. Please enable a GPU runtime in Colab: Runtime > Change runtime type > GPU.')

## 3. Configuration

For the first smoke test, set `N_SAMPLES = 10`. After verifying CSV correctness and GPU stability, increase to 25 or 50. The final requested evaluation size is 50.

In [ ]:
BASE_MODEL_ID = 'mistralai/Mistral-7B-Instruct-v0.3'
LORA_MODEL_ID = 'millat/StudyAbroadGPT-7B-LoRa-Kaggle'
LORA_MODEL_SUBFOLDER = 'merged'
DATASET_ID = 'millat/StudyAbroadGPT-Dataset'

RANDOM_SEED = 42
N_SAMPLES = 50  # Recommended workflow: test with 10, then 25, then 50.

# Deterministic config for scored evaluation.
EVAL_GENERATION_CONFIG = {
    'max_new_tokens': 256,
    'do_sample': False,
    'temperature': 0.0,
    'top_p': 1.0,
}

# Optional qualitative/demo config. Not used for scored CSV unless you explicitly switch configs.
DEMO_GENERATION_CONFIG = {
    'max_new_tokens': 256,
    'do_sample': True,
    'temperature': 0.7,
    'top_p': 0.9,
}

GENERATION_CONFIG = EVAL_GENERATION_CONFIG

random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)

print('Base model:', BASE_MODEL_ID)
print('LoRA/merged model:', LORA_MODEL_ID)
print('LoRA model subfolder:', LORA_MODEL_SUBFOLDER)
print('Dataset:', DATASET_ID)
print('Sample size:', N_SAMPLES)
print('Generation config:', GENERATION_CONFIG)

## 4. Load held-out test prompts

Only the `test` split is used. The notebook samples 50 random conversations using seed 42 and extracts the first user/human prompt from each conversation.

In [ ]:
def first_user_prompt(conversation):
    for turn in conversation:
        if turn.get('from') == 'human':
            return str(turn.get('value', '')).strip()
    return ''

prompts_path = OUTPUT_DIR / 'evaluation_prompts.csv'
existing_prompts = load_csv_if_exists(prompts_path)

if existing_prompts is not None and len(existing_prompts):
    prompts_df = existing_prompts
    log(f'Loaded prompts from: {prompts_path}')
else:
    raw_test = load_dataset(DATASET_ID, split='test')
    rng = random.Random(RANDOM_SEED)
    indices = list(range(len(raw_test)))
    rng.shuffle(indices)
    selected_indices = indices[:N_SAMPLES]

    samples = []
    for sample_id, idx in enumerate(selected_indices, start=1):
        item = raw_test[idx]
        prompt = first_user_prompt(item.get('conversations', []))
        if not prompt:
            continue
        samples.append({
            'sample_id': sample_id,
            'dataset_index': idx,
            'prompt': prompt,
        })

    prompts_df = pd.DataFrame(samples)
    save_df(prompts_df, prompts_path)

if SHOW_PREVIEW:
    display(prompts_df.head())
print('Usable prompts:', len(prompts_df))

## 5. Model loading and generation helpers

The notebook loads one model at a time to reduce T4 VRAM pressure. After generating base responses, the base model is deleted and CUDA cache is cleared before loading the LoRA/merged model.

In [ ]:
def clear_gpu_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def build_quantization_config(model_id: str):
    """Use 4-bit loading. The Unsloth base is already a bnb-4bit repo, but this config is also safe for the merged model when memory is constrained."""
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )


def load_model_and_tokenizer(model_id: str, subfolder: str | None = None):
    if not model_id:
        raise ValueError('model_id is required')
    log('Loading tokenizer...')
    quant_config = build_quantization_config(model_id)
    tokenizer_kwargs = {
        'use_fast': True,
        'trust_remote_code': True,
    }
    if subfolder:
        tokenizer_kwargs['subfolder'] = subfolder
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_id, **tokenizer_kwargs)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        tokenizer.padding_side = 'left'

        log('Loading model...')
        model_kwargs = {
            'device_map': 'auto',
            'torch_dtype': torch.float16,
            'low_cpu_mem_usage': True,
            'quantization_config': quant_config,
            'trust_remote_code': True,
        }
        if subfolder:
            model_kwargs['subfolder'] = subfolder
        model = AutoModelForCausalLM.from_pretrained(model_id, **model_kwargs)
    except OSError as exc:
        if model_id.endswith('-bnb-4bit') and 'does not appear to have a file named' in str(exc):
            fallback_id = model_id.replace('-bnb-4bit', '')
            log('Falling back to base repo.')
            tokenizer = AutoTokenizer.from_pretrained(fallback_id, **tokenizer_kwargs)
            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token
            tokenizer.padding_side = 'left'

            model = AutoModelForCausalLM.from_pretrained(fallback_id, **model_kwargs)
        else:
            raise
    model.eval()
    return model, tokenizer


def format_prompt(tokenizer, prompt: str):
    messages = [{'role': 'user', 'content': prompt}]
    if hasattr(tokenizer, 'apply_chat_template') and tokenizer.chat_template:
        try:
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        except Exception as exc:
            print('Chat template failed, falling back to Mistral INST format:', repr(exc))
    return f'<s>[INST] {prompt.strip()} [/INST]'


def count_tokens(tokenizer, text: str):
    return len(tokenizer(text, add_special_tokens=False).input_ids)


def generate_one(model, tokenizer, prompt: str, generation_config: dict):
    formatted_prompt = format_prompt(tokenizer, prompt)
    inputs = tokenizer(formatted_prompt, return_tensors='pt').to(model.device)
    input_token_count = int(inputs['input_ids'].shape[-1])

    start = time.perf_counter()
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=generation_config['max_new_tokens'],
            do_sample=generation_config['do_sample'],
            temperature=generation_config.get('temperature', None),
            top_p=generation_config.get('top_p', None),
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    elapsed = time.perf_counter() - start

    new_token_ids = output_ids[0][input_token_count:]
    response = tokenizer.decode(new_token_ids, skip_special_tokens=True).strip()
    output_token_count = int(len(new_token_ids))
    was_truncated = output_token_count >= generation_config['max_new_tokens']

    return {
        'response': response,
        'generation_time_sec': elapsed,
        'response_char_length': len(response),
        'response_token_count': output_token_count,
        'was_truncated': was_truncated,
    }


def generate_responses_for_model(model_id: str, prompts: list[str], generation_config: dict, label: str, subfolder: str | None = None):
    clear_gpu_memory()
    model, tokenizer = load_model_and_tokenizer(model_id, subfolder=subfolder)
    results = []
    for prompt in tqdm(prompts, desc=f'Generating {label} responses'):
        result = generate_one(model, tokenizer, prompt, generation_config)
        results.append(result)
    del model
    del tokenizer
    clear_gpu_memory()
    return results

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 6. Generate base model responses

This cell may take several minutes on a T4. If testing the notebook for the first time, set `N_SAMPLES = 10` in the configuration cell.

In [ ]:
prompts = prompts_df['prompt'].tolist()

base_outputs_path = OUTPUT_DIR / 'base_model_outputs.csv'
base_outputs_df = load_csv_if_exists(base_outputs_path)

if base_outputs_df is None or base_outputs_df.empty:
    base_results = generate_responses_for_model(
        BASE_MODEL_ID,
        prompts,
        GENERATION_CONFIG,
        label='base model',
    )
    base_outputs_df = pd.DataFrame([
        {
            'sample_id': row['sample_id'],
            'dataset_index': row['dataset_index'],
            'prompt': row['prompt'],
            'response': base['response'],
            'generation_time_sec': base['generation_time_sec'],
            'response_char_length': base['response_char_length'],
            'response_token_count': base['response_token_count'],
            'was_truncated': base['was_truncated'],
        }
        for row, base in zip(prompts_df.to_dict('records'), base_results)
    ])
    save_df(base_outputs_df, base_outputs_path)
else:
    log(f'Loaded base outputs from: {base_outputs_path}')

base_results = base_outputs_df[[
    'response',
    'generation_time_sec',
    'response_char_length',
    'response_token_count',
    'was_truncated',
]].to_dict('records')
print('Base responses:', len(base_results))

## 7. Generate LoRA/merged model responses

The base model has been unloaded before this cell to reduce GPU memory pressure.

In [ ]:
# Safety fallback: if you updated the notebook but did not rerun earlier cells,
# define the LoRA repo/subfolder and subfolder-aware loading helpers here.
BASE_MODEL_ID = 'mistralai/Mistral-7B-Instruct-v0.3'
LORA_MODEL_ID = 'millat/StudyAbroadGPT-7B-LoRa-Kaggle'
LORA_MODEL_SUBFOLDER = 'merged'

# Always use the LoRA merged weights in this cell.
LORA_MODEL_ID = 'millat/StudyAbroadGPT-7B-LoRa-Kaggle'
LORA_MODEL_SUBFOLDER = 'merged'

prompts_path = OUTPUT_DIR / 'evaluation_prompts.csv'
if 'prompts_df' not in globals() or prompts_df is None:
    prompts_df = load_csv_if_exists(prompts_path)
if prompts_df is None or prompts_df.empty:
    raise RuntimeError('Prompts not found. Run the prompt loading cell first.')
prompts = prompts_df['prompt'].tolist()

# Redefine subfolder-aware helpers in this cell to avoid stale in-memory definitions.
def load_model_and_tokenizer(model_id: str, subfolder: str | None = None, base_model_id: str | None = None):
    if not model_id:
        raise ValueError('model_id is required')
    log('Loading tokenizer...')
    quant_config = build_quantization_config(model_id)
    tokenizer_kwargs = {
        'use_fast': True,
        'trust_remote_code': True,
    }
    if subfolder:
        tokenizer_kwargs['subfolder'] = subfolder

    tokenizer = AutoTokenizer.from_pretrained(model_id, **tokenizer_kwargs)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = 'left'

    log('Loading model...')
    model_kwargs = {
        'device_map': 'auto',
        'dtype': torch.float16,
        'low_cpu_mem_usage': True,
        'quantization_config': quant_config,
        'trust_remote_code': True,
    }
    if subfolder:
        model_kwargs['subfolder'] = subfolder
    try:
        model = AutoModelForCausalLM.from_pretrained(model_id, **model_kwargs)
        model.eval()
        return model, tokenizer
    except OSError as exc:
        if 'does not appear to have a file named' not in str(exc):
            raise
        base_model_id = base_model_id or BASE_MODEL_ID
        try:
            from huggingface_hub import list_repo_files, snapshot_download
        except ImportError as hub_exc:
            raise RuntimeError('huggingface_hub is required for adapter detection.') from hub_exc
        repo_files = list_repo_files(model_id)
        adapter_path = f"{subfolder}/adapter_config.json" if subfolder else 'adapter_config.json'
        if adapter_path not in repo_files:
            if not subfolder:
                raise OSError('LoRA adapter config not found; merged weights should load directly. Recheck the model_id/subfolder.') from exc
            log('Loading merged weights from local snapshot...')
            local_dir = snapshot_download(model_id, allow_patterns=[f"{subfolder}/*"])
            merged_path = Path(local_dir) / subfolder
            model_kwargs.pop('subfolder', None)
            model = AutoModelForCausalLM.from_pretrained(str(merged_path), **model_kwargs)
            model.eval()
            return model, tokenizer
        log('Loading base model for LoRA adapter...')
        base_model_kwargs = {
            'device_map': 'auto',
            'dtype': torch.float16,
            'low_cpu_mem_usage': True,
            'quantization_config': quant_config,
            'trust_remote_code': True,
        }
        base_model = AutoModelForCausalLM.from_pretrained(base_model_id, **base_model_kwargs)
        try:
            from peft import PeftModel
        except ImportError as peft_exc:
            raise RuntimeError('peft is required to load LoRA adapters. Install it with: pip install peft') from peft_exc
        peft_kwargs = {}
        if subfolder:
            peft_kwargs['subfolder'] = subfolder
        model = PeftModel.from_pretrained(base_model, model_id, **peft_kwargs)
        model.eval()
        return model, tokenizer


def generate_responses_for_model(
    model_id: str,
    prompts: list[str],
    generation_config: dict,
    label: str,
    subfolder: str | None = None,
    base_model_id: str | None = None,
    ):
    clear_gpu_memory()
    model, tokenizer = load_model_and_tokenizer(
        model_id,
        subfolder=subfolder,
        base_model_id=base_model_id,
    )
    results = []
    for prompt in tqdm(prompts, desc=f'Generating {label} responses'):
        result = generate_one(model, tokenizer, prompt, generation_config)
        results.append(result)
    del model
    del tokenizer
    clear_gpu_memory()
    return results

lora_outputs_path = OUTPUT_DIR / 'lora_model_outputs.csv'
lora_outputs_df = load_csv_if_exists(lora_outputs_path)

if lora_outputs_df is None or lora_outputs_df.empty:
    lora_results = generate_responses_for_model(
        LORA_MODEL_ID,
        prompts,
        GENERATION_CONFIG,
        label='LoRA/merged model',
        subfolder=LORA_MODEL_SUBFOLDER,
        base_model_id=BASE_MODEL_ID,
    )
    lora_outputs_df = pd.DataFrame([
        {
            'sample_id': row['sample_id'],
            'dataset_index': row['dataset_index'],
            'prompt': row['prompt'],
            'response': lora['response'],
            'generation_time_sec': lora['generation_time_sec'],
            'response_char_length': lora['response_char_length'],
            'response_token_count': lora['response_token_count'],
            'was_truncated': lora['was_truncated'],
        }
        for row, lora in zip(prompts_df.to_dict('records'), lora_results)
    ])
    save_df(lora_outputs_df, lora_outputs_path)
else:
    log(f'Loaded LoRA outputs from: {lora_outputs_path}')

lora_results = lora_outputs_df[[
    'response',
    'generation_time_sec',
    'response_char_length',
    'response_token_count',
    'was_truncated',
]].to_dict('records')
print('LoRA responses:', len(lora_results))

## 8. Build evaluation dataframe and save metadata

This section stores raw prompts, raw responses, generation times, response lengths, token counts, and truncation flags. Truncation flags help identify whether many outputs hit `max_new_tokens`, which can bias helpfulness scoring.

In [ ]:
raw_path = OUTPUT_DIR / 'downstream_raw_model_outputs.csv'
results_df = load_csv_if_exists(raw_path)

if results_df is None or results_df.empty:
    base_outputs_path = OUTPUT_DIR / 'base_model_outputs.csv'
    lora_outputs_path = OUTPUT_DIR / 'lora_model_outputs.csv'
    base_outputs_df = load_csv_if_exists(base_outputs_path)
    lora_outputs_df = load_csv_if_exists(lora_outputs_path)

    if base_outputs_df is None or lora_outputs_df is None:
        raise RuntimeError('Base/LoRA outputs not found. Run the generation cells first.')

    records = []
    for base_row, lora_row in zip(
        base_outputs_df.to_dict('records'),
        lora_outputs_df.to_dict('records'),
    ):
        records.append({
            'sample_id': base_row['sample_id'],
            'dataset_index': base_row['dataset_index'],
            'prompt': base_row['prompt'],
            'base_model_response': base_row['response'],
            'lora_model_response': lora_row['response'],
            'base_generation_time_sec': base_row['generation_time_sec'],
            'lora_generation_time_sec': lora_row['generation_time_sec'],
            'base_response_char_length': base_row['response_char_length'],
            'lora_response_char_length': lora_row['response_char_length'],
            'base_response_token_count': base_row['response_token_count'],
            'lora_response_token_count': lora_row['response_token_count'],
            'base_was_truncated': base_row['was_truncated'],
            'lora_was_truncated': lora_row['was_truncated'],
        })

    results_df = pd.DataFrame(records)
    save_df(results_df, raw_path)
else:
    log(f'Loaded raw model outputs from: {raw_path}')

if SHOW_PREVIEW:
    display(results_df.head(2))

metadata_df = results_df[[
    'sample_id',
    'dataset_index',
    'base_generation_time_sec',
    'lora_generation_time_sec',
    'base_response_char_length',
    'lora_response_char_length',
    'base_response_token_count',
    'lora_response_token_count',
    'base_was_truncated',
    'lora_was_truncated',
]].copy()
metadata_path = OUTPUT_DIR / 'downstream_generation_metadata.csv'
save_df(metadata_df, metadata_path)

config = {
    'base_model_id': BASE_MODEL_ID,
    'lora_model_id': LORA_MODEL_ID,
    'lora_model_subfolder': LORA_MODEL_SUBFOLDER,
    'dataset_id': DATASET_ID,
    'split': 'test',
    'sample_size': int(N_SAMPLES),
    'usable_prompt_count': int(len(results_df)),
    'random_seed': int(RANDOM_SEED),
    'generation_config': GENERATION_CONFIG,
    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'raw_outputs_path': str(raw_path),
    'metadata_path': str(metadata_path),
    'scored_csv_path': str(OUTPUT_DIR / 'downstream_blinded_evaluation.csv'),
}
config_path = OUTPUT_DIR / 'evaluation_config.json'
save_json(config, config_path)

## 9. Automatic sanity metrics

These are descriptive diagnostics only. This notebook intentionally avoids BLEU, ROUGE, perplexity, and MMLU because they are not appropriate for this manual downstream advising comparison.

In [ ]:
sanity = {
    'base_avg_response_char_length': results_df['base_response_char_length'].mean(),
    'lora_avg_response_char_length': results_df['lora_response_char_length'].mean(),
    'base_avg_response_token_count': results_df['base_response_token_count'].mean(),
    'lora_avg_response_token_count': results_df['lora_response_token_count'].mean(),
    'base_avg_generation_time_sec': results_df['base_generation_time_sec'].mean(),
    'lora_avg_generation_time_sec': results_df['lora_generation_time_sec'].mean(),
    'base_truncation_rate': results_df['base_was_truncated'].mean(),
    'lora_truncation_rate': results_df['lora_was_truncated'].mean(),
}

sanity_df = pd.DataFrame([sanity])
display(sanity_df.T.rename(columns={0: 'value'}))

sanity_path = OUTPUT_DIR / 'automatic_sanity_metrics.csv'
sanity_df.to_csv(sanity_path, index=False)
print('Saved sanity metrics to:', sanity_path)

## 10. Create blinded evaluation CSV

For unbiased human scoring, model outputs are randomized into `response_a` and `response_b` per row. Hidden metadata columns record which model produced each response. Keep these hidden from evaluators if possible.

In [ ]:
rng = random.Random(RANDOM_SEED)
blinded_rows = []

for row in results_df.to_dict('records'):
    if rng.random() < 0.5:
        response_a = row['base_model_response']
        response_b = row['lora_model_response']
        source_a = 'base'
        source_b = 'lora'
    else:
        response_a = row['lora_model_response']
        response_b = row['base_model_response']
        source_a = 'lora'
        source_b = 'base'

    blinded_rows.append({
        'sample_id': row['sample_id'],
        'prompt': row['prompt'],
        'response_a': response_a,
        'response_b': response_b,
        'response_a_source': source_a,
        'response_b_source': source_b,
        'domain_accuracy_a': '',
        'domain_accuracy_b': '',
        'helpfulness_a': '',
        'helpfulness_b': '',
        'preferred_response': '',
        'evaluator_notes': '',
    })

blinded_df = pd.DataFrame(blinded_rows)
blinded_path = OUTPUT_DIR / 'downstream_blinded_evaluation.csv'
blinded_df.to_csv(blinded_path, index=False)
print('Saved blinded evaluation CSV to:', blinded_path)
display(blinded_df.head(2))

## 11. Qualitative side-by-side examples

The following examples are not a substitute for blinded scoring. They are included for quick inspection of response style and obvious failure cases.

In [ ]:
N_EXAMPLES = min(5, len(results_df))
for _, row in results_df.head(N_EXAMPLES).iterrows():
    print('=' * 100)
    print(f"Sample {row['sample_id']} | Dataset index: {row['dataset_index']}")
    print('\nPROMPT:\n', row['prompt'])
    print('\nBASE RESPONSE:\n', row['base_model_response'])
    print('\nLoRA RESPONSE:\n', row['lora_model_response'])
    print()

## 12. Manual scoring instructions

Open `/content/outputs/downstream_blinded_evaluation.csv` and manually score each row.

Recommended scoring anchors:

### Domain accuracy, 0–3

| Score | Meaning |
|---:|---|
| 0 | Incorrect, misleading, hallucinated, or unsafe |
| 1 | Partially correct but incomplete or weakly domain-specific |
| 2 | Mostly correct with minor omissions or caveating issues |
| 3 | Correct, domain-specific, actionable, and appropriately caveated |

### Helpfulness, 0–3

| Score | Meaning |
|---:|---|
| 0 | Not useful or likely to confuse the student |
| 1 | Some useful information but vague or incomplete |
| 2 | Useful and mostly actionable, with minor gaps |
| 3 | Highly useful, clear, actionable, and tailored to the question |

### Preferred response

Use one of:

- `A`
- `B`
- `tie`

Do not reveal `response_a_source` and `response_b_source` to human evaluators until after scoring is complete.

## 13. Post-scoring analysis

After manual scoring is complete, rerun this section. It computes mean ± SD for A/B, maps scores back to base/LoRA, computes preference percentages, and reveals blinded mappings.

In [ ]:
scored_path = OUTPUT_DIR / 'downstream_blinded_evaluation.csv'
scored_df = pd.read_csv(scored_path)

score_cols = ['domain_accuracy_a', 'domain_accuracy_b', 'helpfulness_a', 'helpfulness_b']
for col in score_cols:
    scored_df[col] = pd.to_numeric(scored_df[col], errors='coerce')

print('Completed score counts:')
display(scored_df[score_cols].count().to_frame('non_blank_scores'))

print('\nA/B score summary:')
ab_summary = scored_df[score_cols].agg(['mean', 'std', 'count']).T
ab_summary['mean ± SD'] = ab_summary.apply(
    lambda r: 'Pending' if pd.isna(r['mean']) else f"{r['mean']:.2f} ± {r['std']:.2f}",
    axis=1,
)
display(ab_summary)

# Map A/B scores back to model identity.
mapped_rows = []
for _, row in scored_df.iterrows():
    for side in ['a', 'b']:
        source = row[f'response_{side}_source']
        mapped_rows.append({
            'sample_id': row['sample_id'],
            'source': source,
            'domain_accuracy': row[f'domain_accuracy_{side}'],
            'helpfulness': row[f'helpfulness_{side}'],
        })

mapped_df = pd.DataFrame(mapped_rows)
model_summary = mapped_df.groupby('source')[['domain_accuracy', 'helpfulness']].agg(['mean', 'std', 'count'])
display(model_summary)

# Preference analysis.
def map_preference(row):
    pref = str(row.get('preferred_response', '')).strip().lower()
    if pref == 'a':
        return row['response_a_source']
    if pref == 'b':
        return row['response_b_source']
    if pref == 'tie':
        return 'tie'
    return None

scored_df['preferred_model'] = scored_df.apply(map_preference, axis=1)
preference_counts = scored_df['preferred_model'].dropna().value_counts()
preference_percentages = scored_df['preferred_model'].dropna().value_counts(normalize=True) * 100
preference_summary = pd.DataFrame({
    'count': preference_counts,
    'percent': preference_percentages.round(2),
})
print('\nPreference summary:')
display(preference_summary)

if 'lora' in preference_counts.index:
    non_tie = scored_df[scored_df['preferred_model'].isin(['base', 'lora'])]
    lora_win_rate = (non_tie['preferred_model'] == 'lora').mean() * 100 if len(non_tie) else float('nan')
    print(f"LoRA win rate among non-tie preferences: {lora_win_rate:.2f}%")
else:
    print('LoRA win rate pending: no completed LoRA preferences found yet.')

analysis_path = OUTPUT_DIR / 'downstream_post_scoring_analysis.csv'
mapped_df.to_csv(analysis_path, index=False)
print('Saved mapped post-scoring data to:', analysis_path)

revealed_mapping_path = OUTPUT_DIR / 'downstream_blinded_mapping_revealed.csv'
scored_df[['sample_id', 'response_a_source', 'response_b_source', 'preferred_response', 'preferred_model']].to_csv(revealed_mapping_path, index=False)
print('Saved revealed blinded mappings to:', revealed_mapping_path)

## 14. Download outputs from Colab

Use the Colab file browser or run the following cell to zip the outputs folder.

In [ ]:
!cd /content && zip -r studyabroadgpt_lora_evaluation_outputs.zip outputs
print('Created /content/studyabroadgpt_lora_evaluation_outputs.zip')

## 15. Bundle outputs and Colab logs for download

This creates a single zip with `/content/outputs` and any Colab logs found under `/content/.config/logs`.

In [ ]:
from pathlib import Path
import shutil

bundle_root = Path('/content/bundle')
bundle_root.mkdir(parents=True, exist_ok=True)

outputs_src = Path('/content/outputs')
logs_src = Path('/content/.config/logs')

outputs_dst = bundle_root / 'outputs'
logs_dst = bundle_root / 'logs'

if outputs_src.exists():
    if outputs_dst.exists():
        shutil.rmtree(outputs_dst)
    shutil.copytree(outputs_src, outputs_dst)
    print('Bundled outputs:', outputs_src)
else:
    print('No outputs folder found at /content/outputs')

if logs_src.exists():
    if logs_dst.exists():
        shutil.rmtree(logs_dst)
    shutil.copytree(logs_src, logs_dst)
    print('Bundled logs:', logs_src)
else:
    print('No Colab logs folder found at /content/.config/logs')

zip_path = '/content/studyabroadgpt_outputs_and_logs.zip'
shutil.make_archive(zip_path.replace('.zip', ''), 'zip', bundle_root)
print('Created:', zip_path)

# 16. Lightweight Qualitative Evaluation and Final Reporting

## 16.1 Qualitative comparison summary
- This evaluation is exploratory and single-annotator.
- The notebook focuses on downstream behavioral comparison rather than training-time metrics.
- Qualitative analysis complements the structural dataset audit.
- No claim of fully validated factual correctness is made.

In [ ]:
import re
from pathlib import Path

import pandas as pd

FINAL_OUTPUT_DIR = Path('/content/outputs')
FINAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def _load_results_df():
    candidates = [FINAL_OUTPUT_DIR / 'downstream_raw_model_outputs.csv']
    if 'OUTPUT_DIR' in globals():
        candidates.append(Path(OUTPUT_DIR) / 'downstream_raw_model_outputs.csv')
    for path in candidates:
        if path.exists():
            return pd.read_csv(path)
    raise FileNotFoundError('downstream_raw_model_outputs.csv not found in /content/outputs or OUTPUT_DIR.')

def _build_long_df(results_df: pd.DataFrame) -> pd.DataFrame:
    records = []
    for _, row in results_df.iterrows():
        records.append({
            'sample_id': row['sample_id'],
            'dataset_index': row['dataset_index'],
            'model': 'base',
            'prompt': row['prompt'],
            'response': row['base_model_response'],
            'response_char_length': row['base_response_char_length'],
            'response_token_count': row['base_response_token_count'],
        })
        records.append({
            'sample_id': row['sample_id'],
            'dataset_index': row['dataset_index'],
            'model': 'lora',
            'prompt': row['prompt'],
            'response': row['lora_model_response'],
            'response_char_length': row['lora_response_char_length'],
            'response_token_count': row['lora_response_token_count'],
        })
    return pd.DataFrame(records)

def _has_bullets(text: str) -> bool:
    if not isinstance(text, str):
        return False
    return bool(re.search(r'(^|\n)\s*([-*]|\d+[.)])\s+', text))

def _contains_term(text: str, term: str) -> bool:
    if not isinstance(text, str):
        return False
    lower = text.lower()
    needle = term.lower()
    if ' ' in needle:
        return needle in lower
    return re.search(rf"\b{re.escape(needle)}\b", lower) is not None

results_df = _load_results_df()
long_df = _build_long_df(results_df)

caveat_phrases = [
    'official website',
    'check with',
    'verify with',
    'requirements may change',
    'contact the university',
    'consult the embassy',
    'check the university',
    'check with the university',
    'confirm with',
    'confirm details',
    'subject to change',
    'always verify',
    'check the official',
    'check official',
    'please verify',
    'verify on',
    'consult official',
    'confirm on',
    'confirm with the',
    'check the embassy',
    'check with the embassy',
    'verify with the embassy',
    'check with embassy',
    'consult the university',
    'consult the official',
    'check immigration',
    'check visa rules',
    'contact admissions',
    'contact the admissions',
    'contact the program',
    'contact your embassy',
    'contact the embassy',
    'consult the embassy',
    'refer to the official',
    'see the official',
    'see official',
    'check the official website',
    'visit the official website',
    'refer to the official website',
    'confirm with the official website',
    'consult the official website',
    'check the official source',
    'check official sources',
    'verify on the official website',
    'check with the official website',
    'requirements can change',
    'requirements may vary',
    'check requirements',
    'requirements vary',
    'contact the institution',
    'contact the university admissions',
    'contact the admissions office',
    'contact the school',
    'consult the school',
    'check with the school',
    'check with the program',
    'check with the department',
    'verify with the institution',
    'verify with the school',
    'consult the admissions',
    'confirm with admissions',
    'contact admissions office',
    'verify details',
    'verify directly',
    'confirm directly',
    'check directly',
    'subject to updates',
    'subject to updates',
    'subject to change',
    'policies may change',
    'policies can change',
    'rules may change',
    'rules can change',
    'check the policy',
    'check the policies',
    'confirm policies',
    'confirm eligibility',
    'confirm requirements',
    'verify eligibility',
    'verify requirements',
    'check eligibility',
    'check deadlines',
    'confirm deadlines',
    'verify deadlines',
    'check deadlines on',
    'consult the embassy website',
    'check the embassy website',
    'check the government website',
    'check government website',
    'check the immigration website',
    'check immigration website',
    'consult the immigration website',
    'check with immigration',
    'consult immigration',
    'check visa requirements',
    'verify visa requirements',
    'check with the consulate',
    'consult the consulate',
    'check consulate',
    'contact the consulate',
    'contact the embassy',
    'contact your embassy',
    'consult your embassy',
    'contact the visa office',
    'consult the visa office',
    'check the visa office',
    'contact visa office',
    'contact the government',
    'consult the government',
    'check the government',
    'check the official government website',
    'check official government website',
    'official guidance',
    'official guidance may change',
    'official guidance can change',
    'official guidance is subject to change',
    'check official guidance',
    'check official guidance on',
    'consult official guidance',
    'verify official guidance',
    'confirm official guidance',
    'check official requirements',
    'verify official requirements',
    'confirm official requirements',
    'check the official requirements',
    'verify the official requirements',
    'confirm the official requirements',
    'check the latest requirements',
    'verify the latest requirements',
    'confirm the latest requirements',
    'check the latest rules',
    'verify the latest rules',
    'confirm the latest rules',
    'check the latest policy',
    'verify the latest policy',
    'confirm the latest policy',
    'check the latest policies',
    'verify the latest policies',
    'confirm the latest policies',
    'check the latest guidance',
    'verify the latest guidance',
    'confirm the latest guidance',
    'check current requirements',
    'verify current requirements',
    'confirm current requirements',
    'check current policies',
    'verify current policies',
    'confirm current policies',
    'check current rules',
    'verify current rules',
    'confirm current rules',
    'check current guidance',
    'verify current guidance',
    'confirm current guidance',
    'check current information',
    'verify current information',
    'confirm current information',
    'check current details',
    'verify current details',
    'confirm current details',
    'check current status',
    'verify current status',
    'confirm current status',
    'check official updates',
    'verify official updates',
    'confirm official updates',
    'check for updates',
    'verify updates',
    'confirm updates',
    'check updates',
    'verify with official sources',
    'confirm with official sources',
    'check official sources',
    'consult official sources',
    'verify official sources',
    'confirm official sources',
    'check reliable sources',
    'verify reliable sources',
    'confirm reliable sources',
    'check trusted sources',
    'verify trusted sources',
    'confirm trusted sources',
    'check government sources',
    'verify government sources',
    'confirm government sources',
    'check embassy sources',
    'verify embassy sources',
    'confirm embassy sources',
    'check university sources',
    'verify university sources',
    'confirm university sources',
    'check program sources',
    'verify program sources',
    'confirm program sources',
    'check department sources',
    'verify department sources',
    'confirm department sources'
 ]
domain_terms = [
    'visa',
    'scholarship',
    'sop',
    'ielts',
    'accommodation',
    'tuition',
    'embassy',
    'university',
    'admission',
    'recommendation letter',
 ]
risk_terms = [
    'guaranteed',
    'always',
    'definitely approved',
 ]

summary_rows = []
for model in ['base', 'lora']:
    subset = long_df[long_df['model'] == model]
    avg_len = subset['response_char_length'].mean()
    avg_tokens = subset['response_token_count'].mean()
    bullet_pct = subset['response'].apply(_has_bullets).mean() * 100
    caveat_pct = subset['response'].apply(
        lambda text: any(phrase in str(text).lower() for phrase in caveat_phrases)
    ).mean() * 100
    summary_rows.append({
        'model': model,
        'avg_response_char_length': avg_len,
        'avg_response_token_count': avg_tokens,
        'bullet_list_pct': bullet_pct,
        'caveat_phrase_pct': caveat_pct,
    })
summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

domain_rows = []
for model in ['base', 'lora']:
    subset = long_df[long_df['model'] == model]
    for term in domain_terms:
        count = subset['response'].apply(lambda text: _contains_term(text, term)).sum()
        pct = (count / len(subset)) * 100 if len(subset) else 0
        domain_rows.append({
            'model': model,
            'term': term,
            'response_count': int(count),
            'response_pct': round(pct, 2),
        })
domain_df = pd.DataFrame(domain_rows)
domain_output_path = FINAL_OUTPUT_DIR / 'domain_specificity_metrics.csv'
domain_df.to_csv(domain_output_path, index=False)
print(f'Saved domain metrics to: {domain_output_path}')

flag_reasons = {}
def _add_reason(sample_id: int, model: str, reason: str):
    key = (int(sample_id), model)
    flag_reasons.setdefault(key, set()).add(reason)

for model in ['base', 'lora']:
    subset = long_df[long_df['model'] == model].sort_values('response_char_length')
    for _, row in subset.head(3).iterrows():
        _add_reason(row['sample_id'], model, 'shortest_response')
    for _, row in subset.tail(3).iterrows():
        _add_reason(row['sample_id'], model, 'longest_response')
    for _, row in subset.iterrows():
        response_text = str(row['response'])
        lower = response_text.lower()
        if any(phrase in lower for phrase in caveat_phrases):
            _add_reason(row['sample_id'], model, 'caveat_phrase')
        if any(term in lower for term in risk_terms):
            _add_reason(row['sample_id'], model, 'hallucination_risk_language')

flag_rows = []
for (sample_id, model), reasons in flag_reasons.items():
    row = long_df[(long_df['sample_id'] == sample_id) & (long_df['model'] == model)].iloc[0]
    flag_rows.append({
        'sample_id': int(sample_id),
        'dataset_index': int(row['dataset_index']),
        'model': model,
        'prompt': row['prompt'],
        'response': row['response'],
        'response_char_length': row['response_char_length'],
        'response_token_count': row['response_token_count'],
        'reasons': '; '.join(sorted(reasons)),
    })
flags_df = pd.DataFrame(flag_rows).sort_values(['model', 'sample_id'])
flags_output_path = FINAL_OUTPUT_DIR / 'qualitative_flags.csv'
flags_df.to_csv(flags_output_path, index=False)
print(f'Saved qualitative flags to: {flags_output_path}')

## 16.2 Final experimental summary generator and artifact manifest
This section generates the final summary and a manifest of outputs under `/content/outputs`.

In [ ]:
import re
from pathlib import Path

import pandas as pd

FINAL_OUTPUT_DIR = Path('/content/outputs')
FINAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def _find_existing_path(filename: str) -> Path | None:
    candidates = [FINAL_OUTPUT_DIR / filename]
    if 'OUTPUT_DIR' in globals():
        candidates.append(Path(OUTPUT_DIR) / filename)
    for path in candidates:
        if path.exists():
            return path
    return None

def _load_results_df() -> pd.DataFrame:
    path = _find_existing_path('downstream_raw_model_outputs.csv')
    if not path:
        raise FileNotFoundError('downstream_raw_model_outputs.csv not found for summary.')
    return pd.read_csv(path)

def _has_bullets(text: str) -> bool:
    if not isinstance(text, str):
        return False
    return bool(re.search(r'(^|\n)\s*([-*]|\d+[.)])\s+', text))

def _contains_phrase(text: str, phrase: str) -> bool:
    if not isinstance(text, str):
        return False
    return phrase in text.lower()

results_df = _load_results_df()

caveat_phrases = [
    'official website',
    'check with',
    'verify with',
    'requirements may change',
    'contact the university',
    'consult the embassy',
 ]
domain_terms = [
    'visa',
    'scholarship',
    'sop',
    'ielts',
    'accommodation',
    'tuition',
    'embassy',
    'university',
    'admission',
    'recommendation letter',
 ]

base_responses = results_df['base_model_response'].astype(str)
lora_responses = results_df['lora_model_response'].astype(str)

avg_base_len = results_df['base_response_char_length'].mean()
avg_lora_len = results_df['lora_response_char_length'].mean()
length_diff = avg_lora_len - avg_base_len

base_bullets = base_responses.apply(_has_bullets).mean() * 100
lora_bullets = lora_responses.apply(_has_bullets).mean() * 100

base_caveat = base_responses.apply(
    lambda text: any(_contains_phrase(text, phrase) for phrase in caveat_phrases)
).mean() * 100
lora_caveat = lora_responses.apply(
    lambda text: any(_contains_phrase(text, phrase) for phrase in caveat_phrases)
).mean() * 100

domain_rows = []
for model, responses in [('base', base_responses), ('lora', lora_responses)]:
    for term in domain_terms:
        count = responses.apply(lambda text: re.search(rf"\b{re.escape(term)}\b", text.lower()) is not None if ' ' not in term else term in text.lower()).sum()
        pct = (count / len(responses)) * 100 if len(responses) else 0
        domain_rows.append({
            'model': model,
            'term': term,
            'response_count': int(count),
            'response_pct': round(pct, 2),
        })
domain_df = pd.DataFrame(domain_rows)
top_domain = (
    domain_df.sort_values(['model', 'response_pct'], ascending=[True, False])
    .groupby('model')
    .head(3)
)

config_path = _find_existing_path('evaluation_config.json')
config = {}
if config_path:
    config = pd.read_json(config_path, typ='series').to_dict()
else:
    config = {
        'base_model_id': globals().get('BASE_MODEL_ID', 'unknown'),
        'lora_model_id': globals().get('LORA_MODEL_ID', 'unknown'),
        'lora_model_subfolder': globals().get('LORA_MODEL_SUBFOLDER', 'unknown'),
        'dataset_id': globals().get('DATASET_ID', 'unknown'),
        'sample_size': int(len(results_df)),
        'generation_config': globals().get('GENERATION_CONFIG', {}),
    }

blinded_path = _find_existing_path('downstream_blinded_evaluation.csv')
blinded_used = blinded_path is not None
manual_scored = False
if blinded_path:
    scored_df = pd.read_csv(blinded_path)
    score_cols = ['domain_accuracy_a', 'domain_accuracy_b', 'helpfulness_a', 'helpfulness_b']
    for col in score_cols:
        if col in scored_df.columns:
            scored_df[col] = pd.to_numeric(scored_df[col], errors='coerce')
    manual_scored = scored_df[score_cols].notna().any().any() if all(col in scored_df.columns for col in score_cols) else False

summary_lines = [
    '# Final Experimental Summary',
    '',
    '## Models and Data',
    f"- Base model: {config.get('base_model_id', 'unknown')}",
    f"- LoRA model: {config.get('lora_model_id', 'unknown')} (subfolder: {config.get('lora_model_subfolder', 'unknown')})",
    f"- Dataset: {config.get('dataset_id', 'unknown')} (split: {config.get('split', 'test')})",
    f"- Sample size: {config.get('sample_size', len(results_df))}",
    '',
    '## Generation Settings',
    f"- Deterministic generation: {not config.get('generation_config', {}).get('do_sample', False)}",
    f"- Generation config: {config.get('generation_config', {})}",
    '',
    '## Evaluation Setup',
    f"- Blinded evaluation CSV: {'yes' if blinded_used else 'no'}",
    f"- Manual scoring completed: {'yes' if manual_scored else 'no'}",
    '',
    '## Automatic Qualitative Findings',
    f"- Avg response length (chars): base {avg_base_len:.2f}, LoRA {avg_lora_len:.2f} (diff {length_diff:+.2f})",
    f"- Bullet/list usage: base {base_bullets:.1f}%, LoRA {lora_bullets:.1f}%",
    f"- Caveat phrase usage: base {base_caveat:.1f}%, LoRA {lora_caveat:.1f}%",
    '','- Top domain terms by coverage:',
 ]
for model in ['base', 'lora']:
    subset = top_domain[top_domain['model'] == model]
    terms = ', '.join([f"{row['term']} ({row['response_pct']}%)" for _, row in subset.iterrows()])
    summary_lines.append(f"  - {model}: {terms}")

summary_lines.extend([
    '',
    '## Limitations',
    '- No inter-rater agreement is measured.',
    '- No completed factuality verification is included.',
    '- Qualitative findings are exploratory.',
    '- Downstream behavioral improvement is not claimed unless manually validated.',
 ])

summary_path = FINAL_OUTPUT_DIR / 'final_experimental_summary.md'
summary_path.write_text('\n'.join(summary_lines))
print(f'Saved summary to: {summary_path}')

manifest_entries = {
    'evaluation_prompts.csv': 'Held-out prompts used for generation.',
    'base_model_outputs.csv': 'Base model responses and metadata.',
    'lora_model_outputs.csv': 'LoRA/merged model responses and metadata.',
    'downstream_raw_model_outputs.csv': 'Merged base/LoRA outputs and timing metadata.',
    'downstream_generation_metadata.csv': 'Generation-time metadata for both models.',
    'downstream_blinded_evaluation.csv': 'Blinded A/B evaluation sheet for manual scoring.',
    'downstream_blinded_mapping_revealed.csv': 'Revealed A/B mappings after scoring.',
    'automatic_sanity_metrics.csv': 'Length, timing, and truncation diagnostics.',
    'downstream_post_scoring_analysis.csv': 'Post-scoring analysis outputs (if scored).',
    'evaluation_config.json': 'Configuration and reproducibility metadata.',
    'domain_specificity_metrics.csv': 'Domain term coverage metrics (this section).',
    'qualitative_flags.csv': 'Flagged long/short/caveat/risk responses.',
    'final_experimental_summary.md': 'Auto-generated experimental summary.',
    'artifact_manifest.md': 'This manifest of outputs.',
}

manifest_lines = [
    '# Artifact Manifest',
    '',
    'All files listed below are expected outputs. Paths reflect the existing location if found.',
 ]
for filename, description in manifest_entries.items():
    found_path = _find_existing_path(filename)
    path_text = str(found_path) if found_path else str(FINAL_OUTPUT_DIR / filename)
    manifest_lines.append(f"- {filename}: {description} ({path_text})")

manifest_path = FINAL_OUTPUT_DIR / 'artifact_manifest.md'
manifest_path.write_text('\n'.join(manifest_lines))
print(f'Saved manifest to: {manifest_path}')